In [7]:
import pandas as pd
import os

In [8]:
# ==============================================================================
# FUNCTION 1: Split and Filter DataFrame by Region
# ==============================================================================
def split_csv_by_region(coef_csv_path, vif_csv_path, region_prefix):
    """
    Reads raw CSVs, sets 'feature_or_metric' as index, and filters columns.
    Retains the exact row order from the raw CSV.
    """
    df_coef_raw = pd.read_csv(coef_csv_path).set_index('feature_or_metric')
    df_vif_raw = pd.read_csv(vif_csv_path).set_index('feature_or_metric')
    
    scales = ['100m', '250m', '500m', '750m', '1000m']
    target_columns = [f"{region_prefix}_{scale}" for scale in scales]
    
    df_coef_region = df_coef_raw[target_columns].copy()
    df_vif_region = df_vif_raw[target_columns].copy()
    
    rename_dict = {f"{region_prefix}_{scale}": scale for scale in scales}
    df_coef_region.rename(columns=rename_dict, inplace=True)
    df_vif_region.rename(columns=rename_dict, inplace=True)
    
    return df_coef_region, df_vif_region


# ==============================================================================
# FUNCTION 2: Merge Metrics with 3-Decimal Formatting
# ==============================================================================
def merge_metrics_to_dictionary(df_coef, df_vif):
    """
    Combines Coef and VIF into 'Coef*** (VIF)' format with exactly 3 decimals.
    Returns both the data dictionary AND the ordered list of variables from CSV.
    """
    scales = ['100m', '250m', '500m', '750m', '1000m']
    df_merged = df_coef.join(df_vif, lsuffix='_coef', rsuffix='_vif')
    
    # Capture the exact variable names and execution order from CSV index
    ordered_variables = list(df_merged.index)
    
    def format_to_3_decimal(val_str, is_vif=False):
        try:
            clean_str = str(val_str).strip()
            if '*' in clean_str:
                num_part = clean_str.split('*')[0]
                stars_part = clean_str[len(num_part):]
                return f"{float(num_part):.3f}{stars_part}"
            else:
                if is_vif:
                    return f"({float(clean_str):.3f})"
                return f"{float(clean_str):.3f}"
        except ValueError:
            if is_vif:
                return f"({val_str})" if pd.notna(val_str) and str(val_str).lower() != 'nan' else ''
            return str(val_str).strip()

    data_dict = {}
    for var_name, row in df_merged.iterrows():
        data_dict[var_name] = {}
        for scale in scales:
            coef_raw = row[f'{scale}_coef']
            vif_raw = row[f'{scale}_vif']
            
            if var_name in ['R2', 'Features_N', 'Observations_N', '$R^2$']:
                data_dict[var_name][scale] = format_to_3_decimal(coef_raw, is_vif=False)
            else:
                coef_formatted = format_to_3_decimal(coef_raw, is_vif=False)
                if pd.notna(vif_raw) and str(vif_raw).lower() != 'nan':
                    vif_formatted = format_to_3_decimal(vif_raw, is_vif=True)
                    data_dict[var_name][scale] = f"{coef_formatted} {vif_formatted}"
                else:
                    data_dict[var_name][scale] = coef_formatted
                
    return data_dict, ordered_variables


# ==============================================================================
# FUNCTION 3: Fully Dynamic Data-Driven LaTeX Generator
# ==============================================================================
def generate_dynamic_latex_file(data_dict, ordered_variables, region_name, output_tex_path):
    """
    Dynamically loops over the ordered_variables extracted directly from the CSV.
    Eliminates hardcoded variable definitions while formatting LaTeX underscores safely.
    """
    
    # 1. Setup metadata rows and statistical variables separately
    metadata_keys = ['R2', 'Features_N', 'Observations_N', '$R^2$']
    
    # Filter out metadata from the main regression rows
    model_variables = [v for v in ordered_variables if v not in metadata_keys]

    # 2. Build LaTeX Header
    latex_str = (
        "\\begin{table*}[tbp]\n"
        f"\\caption{{Standardized coefficients and Variance Inflation Factors (VIF) of the MLR models across five spatial scales for the \\textbf{{ {region_name} }} dataset. Significance levels are indicated by asterisks, and VIF values are provided in parentheses.}}\\label{{Tab:S1_{region_name.replace(' ', '_')}}}\n"
        "\\begin{tabular*}{\\textwidth}{@{\\extracolsep{\\fill}} l c c c c c @{}}\n"
        "\\toprule\n"
        "\\multirow{2}{*}{Variables} & \\multicolumn{5}{c}{Grid Scale} \\\\\n"
        "\\cmidrule{2-6}\n"
        " & 100m & 250m & 500m & 750m & 1000m \\\\\n"
        "\\midrule\n"
    )

    # 3. Dynamically loop over ALL variables found in the CSV
    for var in model_variables:
        # Automatically escape underscores for LaTeX (e.g., PLAND_built_up -> PLAND\_built\_up)
        latex_label = var.replace('_', '\\_')
        
        d = data_dict[var]
        # Append the formatted data row dynamically
        latex_str += f"{latex_label:<28} &  {d['100m']:<14} &  {d['250m']:<14} &  {d['500m']:<14} &  {d['750m']:<14} &  {d['1000m']:<14} \\\\\n"

    # 4. Append Metadata Rows at the bottom (if they exist in the dataset)
    latex_str += "\\midrule\n"
    
    # Map raw index names to elegant LaTeX display names
    metadata_labels = {
        'R2': '$R^2$',
        '$R^2$': '$R^2$',
        'Features_N': 'Features ($N$)',
        'Observations_N': 'Observations ($N$)'
    }
    
    for meta_key in metadata_keys:
        if meta_key in data_dict:
            d = data_dict[meta_key]
            label = metadata_labels[meta_key]
            latex_str += f"{label:<28} &  {d['100m']:<14} &  {d['250m']:<14} &  {d['500m']:<14} &  {d['750m']:<14} &  {d['1000m']:<14} \\\\\n"

    # 5. Close Table Template
    latex_str += (
        "\\bottomrule\n"
        "\\multicolumn{6}{@{}l}{\\footnotesize Note: *** $p < 0.001$, ** $p < 0.01$, * $p < 0.05$. Values in parentheses are VIFs.}\n"
        "\\end{tabular*}\n"
        "\\end{table*}"
    )

    # Save to file
    os.makedirs(os.path.dirname(output_tex_path) if os.path.dirname(output_tex_path) else '.', exist_ok=True)
    with open(output_tex_path, 'w', encoding='utf-8') as f:
        f.write(latex_str)

In [9]:
# ==============================================================================
# MAIN EXECUTION PIPELINE
# ==============================================================================
if __name__ == "__main__":
    coef_file = "/home/GermanCitiesSUHI/data/results/mlr_results/final_mlr_coef_matrix.csv"
    vif_file = "/home/GermanCitiesSUHI/data/results/mlr_results/final_mlr_vif_matrix.csv"

    regions = {'Global': 'Global', 'Zone_15': 'Zone 15', 'Zone_26': 'Zone 26'}

    for prefix, full_name in regions.items():
        # Step 1: Split
        df_coef_reg, df_vif_reg = split_csv_by_region(coef_file, vif_file, prefix)
        
        # Step 2: Merge and dynamically extract the row index list
        region_dict, csv_variables = merge_metrics_to_dictionary(df_coef_reg, df_vif_reg)
        
        # Step 3: Automatically generate LaTeX using the extracted variable list
        output_path = f"/home/GermanCitiesSUHI/data/results/mlr_results/tables/tab_s1_{prefix.lower()}.tex"
        generate_dynamic_latex_file(region_dict, csv_variables, full_name, output_path)
        
    print("All tables synchronized perfectly with CSV rows!")

All tables synchronized perfectly with CSV rows!


In [13]:
import pandas as pd
import json
import os

# ==============================================================================
# FUNCTION 1: Helper to load hyperparams from external JSON file safely
# ==============================================================================
def load_hyperparams(json_path):
    """
    Reads the external hyperparameter JSON file specified by params_path.
    Maps standard LightGBM keys to their respective ordered list for the LaTeX table.
    """
    # Define fallback defaults if a key is completely missing in the JSON
    defaults = {
        'learning_rate': 0.0, 'num_leaves': 0, 'max_depth': -1, 
        'n_estimators': 0, 'subsample': 1.0, 'colsample_bytree': 1.0, 
        'reg_alpha': 0.0, 'reg_lambda': 0.0, 'min_child_samples': 0
    }
    
    if not isinstance(json_path, str) or pd.isna(json_path) or not os.path.exists(json_path):
        # Return fallback values if the path is corrupted
        return [defaults[k] for k in defaults.keys()]
        
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            p = json.load(f)
            
        # Extract and map values to match the exact columns of the LaTeX table:
        # LR, NL, MD, NE, SS, CS, alpha, lambda, MC
        lr    = p.get('learning_rate', p.get('lr', defaults['learning_rate']))
        nl    = p.get('num_leaves', defaults['num_leaves'])
        md    = p.get('max_depth', defaults['max_depth'])
        ne    = p.get('n_estimators', defaults['n_estimators'])
        ss    = p.get('subsample', defaults['subsample'])
        cs    = p.get('colsample_bytree', defaults['colsample_bytree'])
        alpha = p.get('reg_alpha', defaults['reg_alpha'])
        reg_l = p.get('reg_lambda', defaults['reg_lambda'])
        mc    = p.get('min_child_samples', defaults['min_child_samples'])
        
        return [lr, nl, md, ne, ss, cs, alpha, reg_l, mc]
    except Exception:
        # Return fallback defaults if the JSON structure is broken
        return [defaults[k] for k in defaults.keys()]


# ==============================================================================
# FUNCTION 2: Compile Data Matrix into LaTeX Structure
# ==============================================================================
def generate_lgbm_param_table(summary_csv_path, output_tex_path):
    """
    Parses the main summary CSV, resolves external JSON parameter references,
    formats numeric scales safely, and outputs a highly polished academic LaTeX block.
    """
    df = pd.read_csv(summary_csv_path)
    
    # Pre-defined strict order for rows as required by your manuscript layout
    target_partitions = ['Global', 'Zone_15', 'Zone_26']
    target_scales = [100, 250, 500, 750, 1000]
    
    # Format mapping for formatting columns with proper academic decimal trailing
    # Performance metrics: 3 decimals. Hyperparams: mix of floats (2 dec) and ints.
    def fmt(val):
        if isinstance(val, int) or (isinstance(val, float) and val.is_integer()):
            return str(int(val))
        if isinstance(val, float):
            return f"{val:.3f}" if val < 1.0 and val > 0.0 else f"{val:.2f}"
        return str(val)

    # 1. Build LaTeX Structural Header
    latex_str = (
        "\\begin{table*}[htbp]\n"
        "\\caption{Performance metrics and optimal hyperparameters for the 15 LightGBM models across different spatial scales and climatic partitions.}\\label{Tab:S_LGBM_Params}\n"
        "\\footnotesize % Reduced size to perfectly fit columns within textwidth boundary\n"
        "\\begin{tabular*}{\\textwidth}{@{\\extracolsep{\\fill}} l c ccc ccccccccc @{}}\n"
        "\\toprule\n"
        "\\multirow{2}{*}{Partition} & \\multirow{2}{*}{Scale (m)} & \\multicolumn{3}{c}{Model Performance} & \\multicolumn{9}{c}{Optimal Hyperparameters} \\\\\n"
        "\\cmidrule(lr){3-5} \\cmidrule(l){6-14}\n"
        " & & $R^2$ & RMSE & MAE & LR & NL & MD & NE & SS & CS & $\\alpha$ & $\\lambda$ & MC \\\\\n"
        "\\midrule\n"
    )

    # 2. Iterate through Partitions and Scales dynamically
    for p_idx, partition in enumerate(target_partitions):
        # Display name adjustment (e.g., 'Zone_15' to 'Zone 15')
        display_partition = partition.replace('_', ' ')
        
        # Build the \multirow marker for the first scale of the block
        latex_str += f"% --- {display_partition} Block ---\n"
        latex_str += f"\\multirow{{5}}{{*}}{{\\textbf{{{display_partition}}}}}\n"
        
        for s_idx, scale in enumerate(target_scales):
            # Locate the exact model run from the summary spreadsheet
            row_match = df[(df['partition'] == partition) & (df['scale_m'] == scale)]
            
            if not row_match.empty:
                row = row_match.iloc[0]
                # Extract internal performance metrics
                r2   = f"{row['r2']:.3f}"
                rmse = f"{row['rmse']:.3f}"
                mae  = f"{row['mae']:.3f}"
                
                # Fetch external hyperparameters from file path
                h_params = load_hyperparams(row['params_path'])
                h_formatted = [fmt(val) for val in h_params]
            else:
                # Fallback placeholders if a specific scale-partition combination failed/missing
                r2, rmse, mae = "Missing", "Missing", "Missing"
                h_formatted = ["-"] * 9
            
            # Format rows: The first row needs a spacer after multirow, subsequent rows need an empty lead cell
            lead_space = " " if s_idx == 0 else " "
            param_string = " & ".join(h_formatted)
            
            latex_str += f"{lead_space} & {scale:<4}  & {r2} & {rmse} & {mae} & {param_string} \\\\\n"
        
        # Add separating structural line between blocks (omit after the last block)
        if p_idx < len(target_partitions) - 1:
            latex_str += "\\midrule\n"

    # 3. Append LaTeX Structural Footer and Notes
    latex_str += (
        "\\bottomrule\n"
        "\\multicolumn{14}{@{}l}{\\scriptsize \\textbf{Note}: LR = \\texttt{learning rate}, NL = \\texttt{num leaves}, MD = \\texttt{max depth} (-1 denotes no limit), NE = \\texttt{n estimators}, SS = \\texttt{subsample},} \\\\\n"
        "\\multicolumn{14}{@{}l}{\\scriptsize CS = \\texttt{colsample bytree}, $\\alpha$ = \\texttt{reg alpha} (L1 regularization), $\\lambda$ = \\texttt{reg lambda} (L2 regularization), MC = \\texttt{min child samples}.}\n"
        "\\end{tabular*}\n"
        "\\end{table*}"
    )

    # 4. Write output to standalone .tex file
    with open(output_tex_path, 'w', encoding='utf-8') as f:
        f.write(latex_str)
    print(f"Pipeline completed successfully. Table exported to: {output_tex_path}")

In [14]:
# ==============================================================================
# PIPELINE INVOCATION
# ==============================================================================
if __name__ == "__main__":
    # Path to your master model results tracking table
    summary_csv = "/home/GermanCitiesSUHI/data/results/lgbm_results_selected/final_lgbm_results_vif_selected.csv" 
    output_tex = "/home/GermanCitiesSUHI/data/results/lgbm_results_selected/tab_s_lgbm_params.tex"
    
    generate_lgbm_param_table(summary_csv, output_tex)

Pipeline completed successfully. Table exported to: /home/GermanCitiesSUHI/data/results/lgbm_results_selected/tab_s_lgbm_params.tex
